# Colab de transformaciones homogéneas 3D
En este notebook puedes encontrar un ejemplo práctico de transformaciones.
Puede hacer otros armando sus transformadas :)

In [3]:
%%capture
# Install necessary libraries
!pip install ipywidgets
!pip install matplotlib

In [4]:
from ipywidgets import interact, interactive, fixed, interact_manual
import ipywidgets as widgets

In [5]:
# transformaciones
import numpy as np

class Transformations:

    def translation_transform(self,x, y, z):

        return np.array([[1, 0, 0, x], [0, 1, 0, y], [0, 0, 1, z], [0, 0, 0, 1]])


    def rot_x(self, qx):

        return np.array([[1, 0, 0, 0], [0, np.cos(qx), -np.sin(qx), 0], [0, np.sin(qx), np.cos(qx), 0], [0, 0, 0, 1]])


    def rot_y(self, qy):

        return np.array([[np.cos(qy), 0, np.sin(qy), 0], [0, 1, 0, 0], [-np.sin(qy), 0, np.cos(qy), 0], [0, 0, 0, 1]])


    def rot_z(self, qz):

        return np.array([[np.cos(qz), -np.sin(qz), 0, 0], [np.sin(qz), np.cos(qz), 0, 0], [0, 0, 1, 0], [0, 0, 0, 1]])

In [6]:
import matplotlib.pyplot as plt

class PoseGenerator:
    def __init__(self):
        self.fig=None
        self.fig3D = None
        self.ax=None
        self.ax3D=None

    def create_fig3D(self,nombre="Poses 3D"):
            self.fig3D = plt.figure(figsize=(10, 6))
            self.ax3D = self.fig3D.add_subplot(111, projection='3d')
            self.ax3D.axis('equal')
            self.ax3D.set_xlim([0,600])
            self.ax3D.set_ylim([-210,210])
            self.ax3D.set_zlim([-210,210])
            self.ax3D.set_title(nombre)
            return self.ax3D, self.fig3D

    def show_fig(self):
        plt.show()

    def draw_axes_tf(self, pose, name="", color="k",marker_length=50):
                origin_pose = np.transpose(pose)[3, 0:3]
                x_rot = np.linalg.multi_dot([pose, [1, 0, 0, 0]])
                y_rot = np.linalg.multi_dot([pose, [0, 1, 0, 0]])
                z_rot = np.linalg.multi_dot([pose, [0, 0, 1, 0]])
                self.ax3D.quiver(origin_pose[0], origin_pose[1], origin_pose[2], x_rot[0],
                        x_rot[1], x_rot[2], length=marker_length, normalize=True, color='r')
                self.ax3D.quiver(origin_pose[0], origin_pose[1], origin_pose[2], y_rot[0],
                        y_rot[1], y_rot[2], length=marker_length, normalize=True, color='g')
                self.ax3D.quiver(origin_pose[0], origin_pose[1], origin_pose[2], z_rot[0],
                        z_rot[1], z_rot[2], length=marker_length, normalize=True, color='b')
                self.ax3D.scatter(xs=[origin_pose[0]], ys=[origin_pose[1]],zs=[origin_pose[2]], marker='o', label=name)
                self.ax3D.legend(loc='upper right', bbox_to_anchor=(1.5, 1))

## Manipulador SCARA

<img src="https://raw.githubusercontent.com/uwulises/ME3250_tareas_laboratorios/refs/heads/main/scara.png" width="500">

In [7]:
'''#ejemplo scara
-definir Largos de cada link
-definir angulos iniciales por defecto'''
def SCARA(q1=0,q2=0):

  L1 = 100 #mm
  L2 = 100 #mm
  TF = Transformations()
  Origen = np.identity(4) #pose de origen
  #creamos las transformadas
  T_L1 = TF.translation_transform(L1,0,0)
  T_L2 = TF.translation_transform(L2,0,0)
  Rz_1 = TF.rot_z(q1)
  Rz_2 = TF.rot_z(q2)
  q1_pose = np.linalg.multi_dot([Origen,Rz_1])
  q2_pose = np.linalg.multi_dot([Origen,Rz_1,T_L1,Rz_2])

  M_o_n =  np.linalg.multi_dot([Origen,Rz_1,T_L1,Rz_2,T_L2])
  M_o_n = np.round(M_o_n,2)

  #plot de poses
  PG = PoseGenerator()
  PG.create_fig3D(nombre="SCARA")
  PG.draw_axes_tf(Origen,"Pose origen")
  PG.draw_axes_tf(q1_pose,"q1 Pose")
  PG.draw_axes_tf(q2_pose,"q2 Pose")
  PG.draw_axes_tf(M_o_n,"Tip Pose")
  PG.show_fig()
  print(M_o_n)

#interact(plot_scara, theta1=(-np.pi, np.pi, 0.1), theta2=(-np.pi, np.pi, 0.1))

interact(SCARA,q1=(0, np.pi, 0.1),q2=(-np.pi/2, np.pi/2, 0.1))





interactive(children=(FloatSlider(value=0.0, description='q1', max=3.141592653589793), FloatSlider(value=0.0, …

<function __main__.SCARA(q1=0, q2=0)>

## Manipulador MK2: Este manipulador cuenta con 3 grados de libertad dados por 3 actuadores tipo servomotor.

<img src="https://raw.githubusercontent.com/uwulises/ME3250_tareas_laboratorios/refs/heads/main/mk2.png" width="500">

In [12]:
def MK2(q0=90,q1=90,q2=90, L1=135.0,L2=147.0, a1=100.0, eff_off_x=66.3,eff_off_z=0.0):
  #deg to rad
  q0 = q0*np.pi/180
  q1 = q1*np.pi/180
  q2 = q2*np.pi/180

  TF = Transformations()
  Origen = np.identity(4) #pose de origen
  #creamos las transformadas
  T_a1 = TF.translation_transform(0,0,a1)
  T_L1 = TF.translation_transform(0,0,L1)
  T_L2 = TF.translation_transform(0,0,-L2)
  T_tool = TF.translation_transform(eff_off_x,0,-eff_off_z)

  #creamos las rotaciones
  Rz_0 = TF.rot_z(np.pi/4-q0/2)
  Ry_1 = TF.rot_y(q1-np.pi/2)
  Rz_servo = TF.rot_z(np.pi) #compensacion de sentido del servo
  Ry_2 = TF.rot_y(q1+q2-np.pi/2)
  #compensar Ry_1 , Ry_2
  R_wrist = np.linalg.inv(np.linalg.multi_dot([Ry_1,Rz_servo,Ry_2]))
  #creamos las poses
  q0_pose = np.linalg.multi_dot([Origen,Rz_0])
  q1_pose = np.linalg.multi_dot([q0_pose,T_a1,Ry_1])
  q2_pose = np.linalg.multi_dot([q1_pose,T_L1,Rz_servo,Ry_2])
  wrist_pose = np.linalg.multi_dot([q2_pose,T_L2,R_wrist])
  tip_pose = np.linalg.multi_dot([wrist_pose,T_tool])

  M_o_n =  tip_pose
  M_o_n = np.round(M_o_n,2)

  #plot de poses
  PG = PoseGenerator()
  PG.create_fig3D(nombre="MK2")
  PG.draw_axes_tf(Origen,"Pose origen")
  PG.draw_axes_tf(q0_pose,"q0 Pose")
  PG.draw_axes_tf(q1_pose,"q1 Pose")
  PG.draw_axes_tf(q2_pose,"q2 Pose")
  PG.draw_axes_tf(wrist_pose,"Wrist Pose")
  PG.draw_axes_tf(M_o_n,"Tip Pose")
  PG.show_fig()
  print("Matriz resultante desde la base al efector: \n",M_o_n)
  print("\n")
  print("Coordenadas X,Y,Z:",np.round(np.transpose(M_o_n)[3,0:3],2))
  print("\n")
  return M_o_n

interact(MK2,q0=(0, 180, 1),q1=(0, 180, 1),q2=(0, 180, 1), eff_off_x=(0, 200, 1),eff_off_z=(0, 200, 1), L1=fixed(135.0), L2=fixed(147.0), a1=fixed(100.0))


interactive(children=(IntSlider(value=90, description='q0', max=180), IntSlider(value=90, description='q1', ma…

<function __main__.MK2(q0=90, q1=90, q2=90, L1=135.0, L2=147.0, a1=100.0, eff_off_x=66.3, eff_off_z=0.0)>